In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [2]:
# Load the raw data
raw_data = pd.read_csv('2_dirty_cafe_sales (1).csv')

print("Dataset Shape:", raw_data.shape)
print("\nFirst 10 rows:")
print(raw_data.head(10))
print("\nData Types:")
print(raw_data.dtypes)
print("\nMissing Values:")
print(raw_data.isnull().sum())
print("\nData Quality Issues (checking for 'ERROR' and 'UNKNOWN'):")
for col in raw_data.columns:
    errors = (raw_data[col] == 'ERROR').sum()
    unknowns = (raw_data[col] == 'UNKNOWN').sum()
    if errors > 0 or unknowns > 0:
        print(f"{col}: {errors} ERROR values, {unknowns} UNKNOWN values")

Dataset Shape: (10000, 8)

First 10 rows:
  Transaction ID      Item Quantity Price Per Unit Total Spent  \
0    TXN_1961373    Coffee        2            2.0         4.0   
1    TXN_4977031      Cake        4            3.0        12.0   
2    TXN_4271903    Cookie        4            1.0       ERROR   
3    TXN_7034554     Salad        2            5.0        10.0   
4    TXN_3160411    Coffee        2            2.0         4.0   
5    TXN_2602893  Smoothie        5            4.0        20.0   
6    TXN_4433211   UNKNOWN        3            3.0         9.0   
7    TXN_6699534  Sandwich        4            4.0        16.0   
8    TXN_4717867       NaN        5            3.0        15.0   
9    TXN_2064365  Sandwich        5            4.0        20.0   

   Payment Method  Location Transaction Date  
0     Credit Card  Takeaway       2023-09-08  
1            Cash  In-store       2023-05-16  
2     Credit Card  In-store       2023-07-19  
3         UNKNOWN   UNKNOWN       2023-04-2

In [3]:
# Create a copy for cleaning
df_clean = raw_data.copy()

print(f"Starting with {len(df_clean)} rows\n")

# Step 1: Remove rows with 'ERROR' values
rows_before = len(df_clean)
df_clean = df_clean[~df_clean.astype(str).apply(lambda x: (x == 'ERROR').any(), axis=1)]
print(f"After removing ERROR rows: {len(df_clean)} rows (removed {rows_before - len(df_clean)})")

# Step 2: Remove rows with 'UNKNOWN' values
rows_before = len(df_clean)
df_clean = df_clean[~df_clean.astype(str).apply(lambda x: (x == 'UNKNOWN').any(), axis=1)]
print(f"After removing UNKNOWN rows: {len(df_clean)} rows (removed {rows_before - len(df_clean)})")

# Step 3: Remove rows with missing critical fields
rows_before = len(df_clean)
df_clean = df_clean.dropna(subset=['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date'])
print(f"After removing rows with missing critical fields: {len(df_clean)} rows (removed {rows_before - len(df_clean)})")

# Step 4: Convert data types
df_clean['Quantity'] = pd.to_numeric(df_clean['Quantity'], errors='coerce')
df_clean['Price Per Unit'] = pd.to_numeric(df_clean['Price Per Unit'], errors='coerce')
df_clean['Total Spent'] = pd.to_numeric(df_clean['Total Spent'], errors='coerce')
df_clean['Transaction Date'] = pd.to_datetime(df_clean['Transaction Date'], errors='coerce')

# Remove rows where numeric conversion failed
rows_before = len(df_clean)
df_clean = df_clean.dropna(subset=['Quantity', 'Price Per Unit', 'Total Spent', 'Transaction Date'])
print(f"After removing rows with invalid numeric/date values: {len(df_clean)} rows (removed {rows_before - len(df_clean)})")

# Step 5: Validate Total Spent = Quantity × Price Per Unit (with small tolerance for rounding)
df_clean['Calculated Total'] = df_clean['Quantity'] * df_clean['Price Per Unit']
df_clean['Total Valid'] = np.isclose(df_clean['Total Spent'], df_clean['Calculated Total'], rtol=0.01)
rows_before = len(df_clean)
df_clean = df_clean[df_clean['Total Valid']]
df_clean = df_clean.drop(['Calculated Total', 'Total Valid'], axis=1)
print(f"After validating Total Spent calculations: {len(df_clean)} rows (removed {rows_before - len(df_clean)})")

# Step 6: Remove duplicate transactions
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['Transaction ID'])
print(f"After removing duplicates: {len(df_clean)} rows (removed {rows_before - len(df_clean)})")

print(f"\n✓ Cleaned dataset: {len(df_clean)} rows ({len(df_clean)/len(raw_data)*100:.1f}% of original)")
print(f"✓ Removed: {len(raw_data) - len(df_clean)} rows ({(len(raw_data) - len(df_clean))/len(raw_data)*100:.1f}%)")

Starting with 10000 rows

After removing ERROR rows: 8480 rows (removed 1520)
After removing UNKNOWN rows: 7155 rows (removed 1325)
After removing rows with missing critical fields: 3089 rows (removed 4066)
After removing rows with invalid numeric/date values: 3089 rows (removed 0)
After validating Total Spent calculations: 3089 rows (removed 0)
After removing duplicates: 3089 rows (removed 0)

✓ Cleaned dataset: 3089 rows (30.9% of original)
✓ Removed: 6911 rows (69.1%)
